# External Model Evaluation

This notebook evaluates the finalized DistilBERT comment moderation model on the labeled test data from the Jigsaw Toxic Comment Classification Challenge.

The model, category thresholds, and moderation policy were selected before this external evaluation. No model training or threshold tuning will be performed using this test set.

The Kaggle test labels contain -1 for comments that were not included in the scored test set. These rows are excluded from evaluation.

In [2]:
import pandas as pd

test = pd.read_csv("../data/test.csv")
test_labels = pd.read_csv("../data/test_labels.csv")

print("Test shape:", test.shape)
print("Test labels shape:", test_labels.shape)

Test shape: (153164, 2)
Test labels shape: (153164, 7)


In [3]:
label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

valid_labels = test_labels[
    (test_labels[label_columns] != -1).all(axis=1)
].copy()

print("Usable labeled rows:", len(valid_labels))

Usable labeled rows: 63978


In [4]:
external_test = test.merge(
    valid_labels,
    on="id",
    how="inner"
)

print("External test shape:", external_test.shape)
print()
print("Columns:")
print(external_test.columns.tolist())

External test shape: (63978, 8)

Columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [5]:
print("Duplicate IDs:", external_test["id"].duplicated().sum())
print("Missing comments:", external_test["comment_text"].isna().sum())
print("Missing labels:", external_test[label_columns].isna().sum().sum())

Duplicate IDs: 0
Missing comments: 0
Missing labels: 0


In [6]:
label_counts = external_test[label_columns].sum()

print("Positive comments by category:")
print(label_counts)

Positive comments by category:
toxic            6090
severe_toxic      367
obscene          3691
threat            211
insult           3427
identity_hate     712
dtype: int64


In [7]:
external_test["actual_harmful"] = (
    external_test[label_columns].sum(axis=1) > 0
).astype(int)

harmful_count = external_test["actual_harmful"].sum()
non_harmful_count = len(external_test) - harmful_count

print("Harmful comments:", harmful_count)
print("Non-harmful comments:", non_harmful_count)
print(f"Harmful rate: {harmful_count / len(external_test):.2%}")

Harmful comments: 6243
Non-harmful comments: 57735
Harmful rate: 9.76%


## External Test Set Distribution

After removing rows with unavailable labels (`-1`), the external evaluation set contains 63,978 comments.

- 6,243 comments contain at least one harmful-content label.
- 57,735 comments contain no harmful-content labels.
- The overall harmful-comment rate is 9.76%.

The dataset is therefore highly imbalanced. Performance will be evaluated using precision, recall, F1-score, and confusion matrices rather than relying on accuracy alone. Performance will be examined both at the individual category level and at the overall moderation level.

## Frozen Model Inference

The finalized DistilBERT model is loaded without any additional training. Predictions are generated using batched inference on the external test comments.

Apple Metal Performance Shaders (MPS) is used to accelerate inference on the local GPU.

In [8]:
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_PATH = Path("../models/distilbert_model")

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model.to(device)
model.eval()

print("Device:", device)
print("Model:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

/Users/sphoorthireddymunnangi/Documents/social-media-comment-moderator/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 18:45:25.989000 7148 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7046.63it/s]


Device: mps
Model: DistilBertForSequenceClassification
Number of labels: 6
Label mapping: {0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}


In [9]:
sample_texts = external_test["comment_text"].head(32).tolist()

inputs = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

sample_probabilities = torch.sigmoid(outputs.logits).cpu().numpy()

print("Number of comments:", len(sample_texts))
print("Probability array shape:", sample_probabilities.shape)
print()
print("First comment probabilities:")
print(sample_probabilities[0])

Number of comments: 32
Probability array shape: (32, 6)

First comment probabilities:
[2.5266639e-04 1.2717048e-05 7.8407975e-05 4.6214354e-05 7.5909826e-05
 4.2135398e-05]


In [10]:
import time


test_texts = external_test["comment_text"].head(512).tolist()

start_time = time.time()

inputs = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

# Process the 512 comments in batches of 32
batch_probabilities = []

for start in range(0, len(test_texts), 32):
    batch_texts = test_texts[start:start + 32]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.sigmoid(outputs.logits).cpu().numpy()
    batch_probabilities.append(probabilities)

elapsed_time = time.time() - start_time

print("Comments processed:", len(test_texts))
print(f"Time: {elapsed_time:.2f} seconds")
print(f"Comments per second: {len(test_texts) / elapsed_time:.2f}")

Comments processed: 512
Time: 8.59 seconds
Comments per second: 59.58


In [11]:
import numpy as np
import time


all_texts = external_test["comment_text"].tolist()

batch_size = 32
all_probabilities = []

start_time = time.time()

for start in range(0, len(all_texts), batch_size):
    batch_texts = all_texts[start:start + batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.sigmoid(
        outputs.logits
    ).cpu().numpy()

    all_probabilities.append(probabilities)

    if start % 3200 == 0:
        print(
            f"Processed {min(start + batch_size, len(all_texts)):,} "
            f"of {len(all_texts):,} comments"
        )

external_probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

elapsed_time = time.time() - start_time

print()
print("Finished.")
print("Probability shape:", external_probabilities.shape)
print(f"Total time: {elapsed_time / 60:.2f} minutes")

Processed 32 of 63,978 comments
Processed 3,232 of 63,978 comments
Processed 6,432 of 63,978 comments
Processed 9,632 of 63,978 comments
Processed 12,832 of 63,978 comments
Processed 16,032 of 63,978 comments
Processed 19,232 of 63,978 comments
Processed 22,432 of 63,978 comments
Processed 25,632 of 63,978 comments
Processed 28,832 of 63,978 comments


RuntimeError: MPS backend out of memory (MPS allocated: 1.01 GiB, other allocations: 8.00 GiB, max allowed: 9.07 GiB). Tried to allocate 96.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [12]:
processed_so_far = sum(
    batch.shape[0]
    for batch in all_probabilities
)

print("Predictions still in memory:", processed_so_far)
print("Batches completed:", len(all_probabilities))

Predictions still in memory: 30176
Batches completed: 943


In [13]:
partial_probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

np.save(
    "../data/external_test_probabilities_partial.npy",
    partial_probabilities
)

print("Checkpoint shape:", partial_probabilities.shape)
print("Checkpoint saved.")

Checkpoint shape: (30176, 6)
Checkpoint saved.


In [14]:
import gc

gc.collect()
torch.mps.empty_cache()

print("Unused MPS cache cleared.")

Unused MPS cache cleared.


In [15]:
resume_start = 30176
batch_size = 16

remaining_probabilities = []

resume_time = time.time()

for start in range(resume_start, len(all_texts), batch_size):
    batch_texts = all_texts[start:start + batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.sigmoid(
        outputs.logits
    ).cpu().numpy()

    remaining_probabilities.append(probabilities)

    # Periodically release unused cached GPU memory
    if len(remaining_probabilities) % 100 == 0:
        torch.mps.empty_cache()

        processed = min(
            start + batch_size,
            len(all_texts)
        )

        print(
            f"Processed {processed:,} "
            f"of {len(all_texts):,} comments"
        )

resume_elapsed = time.time() - resume_time

print()
print("Remaining inference finished.")
print(
    "New predictions:",
    sum(batch.shape[0] for batch in remaining_probabilities)
)
print(f"Resume time: {resume_elapsed / 60:.2f} minutes")

RuntimeError: MPS backend out of memory (MPS allocated: 1.01 GiB, other allocations: 8.01 GiB, max allowed: 9.07 GiB). Tried to allocate 48.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [1]:
print("Old model still in memory:", "model" in globals())
print("Old predictions still in memory:", "all_probabilities" in globals())

Old model still in memory: False
Old predictions still in memory: False


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


# Reload external test data
test = pd.read_csv("../data/test.csv")
test_labels = pd.read_csv("../data/test_labels.csv")

label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

valid_labels = test_labels[
    (test_labels[label_columns] != -1).all(axis=1)
].copy()

external_test = test.merge(
    valid_labels,
    on="id",
    how="inner"
)

all_texts = external_test["comment_text"].tolist()


# Reload saved checkpoint
partial_probabilities = np.load(
    "../data/external_test_probabilities_partial.npy"
)


# Reload frozen DistilBERT model
MODEL_PATH = Path("../models/distilbert_model")

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model.to(device)
model.eval()


print("Device:", device)
print("External comments:", len(all_texts))
print("Saved predictions:", partial_probabilities.shape)
print("Resume position:", len(partial_probabilities))

/Users/sphoorthireddymunnangi/Documents/social-media-comment-moderator/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 19:30:57.833000 8122 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8546.89it/s]


Device: mps
External comments: 63978
Saved predictions: (30176, 6)
Resume position: 30176


In [3]:
resume_start = len(partial_probabilities)

test_batch = all_texts[
    resume_start:resume_start + 8
]

inputs = tokenizer(
    test_batch,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

test_probabilities = torch.sigmoid(
    outputs.logits
).cpu().numpy()

print("Starting row:", resume_start)
print("Comments tested:", len(test_batch))
print("Prediction shape:", test_probabilities.shape)

Starting row: 30176
Comments tested: 8
Prediction shape: (8, 6)


In [4]:
del inputs, outputs
torch.mps.empty_cache()

print("Test batch cleared.")

Test batch cleared.


In [5]:
import gc
import time


batch_size = 8
checkpoint_every = 100  # 100 batches × 8 comments = 800 comments

resume_start = len(partial_probabilities)
new_probabilities = []

start_time = time.time()

print(f"Resuming from comment {resume_start:,}")
print(f"Comments remaining: {len(all_texts) - resume_start:,}")
print()

for batch_number, start in enumerate(
    range(resume_start, len(all_texts), batch_size),
    start=1
):
    batch_texts = all_texts[start:start + batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.sigmoid(
        outputs.logits
    ).cpu().numpy()

    new_probabilities.append(probabilities)

    # Remove GPU tensors from this batch
    del inputs, outputs

    # Periodically clear unused MPS memory
    if batch_number % 25 == 0:
        torch.mps.empty_cache()

    # Save progress every 800 comments
    if batch_number % checkpoint_every == 0:
        partial_probabilities = np.concatenate(
            [partial_probabilities] + new_probabilities,
            axis=0
        )

        np.save(
            "../data/external_test_probabilities_partial.npy",
            partial_probabilities
        )

        new_probabilities = []

        gc.collect()
        torch.mps.empty_cache()

        print(
            f"Checkpoint saved: "
            f"{len(partial_probabilities):,} "
            f"of {len(all_texts):,}"
        )


# Save any final batches that did not reach a full checkpoint
if new_probabilities:
    partial_probabilities = np.concatenate(
        [partial_probabilities] + new_probabilities,
        axis=0
    )

    np.save(
        "../data/external_test_probabilities_partial.npy",
        partial_probabilities
    )

elapsed_time = time.time() - start_time

print()
print("Inference finished.")
print("Final checkpoint shape:", partial_probabilities.shape)
print(f"Resume time: {elapsed_time / 60:.2f} minutes")

Resuming from comment 30,176
Comments remaining: 33,802

Checkpoint saved: 30,976 of 63,978
Checkpoint saved: 31,776 of 63,978
Checkpoint saved: 32,576 of 63,978
Checkpoint saved: 33,376 of 63,978
Checkpoint saved: 34,176 of 63,978
Checkpoint saved: 34,976 of 63,978
Checkpoint saved: 35,776 of 63,978
Checkpoint saved: 36,576 of 63,978
Checkpoint saved: 37,376 of 63,978
Checkpoint saved: 38,176 of 63,978
Checkpoint saved: 38,976 of 63,978
Checkpoint saved: 39,776 of 63,978
Checkpoint saved: 40,576 of 63,978
Checkpoint saved: 41,376 of 63,978
Checkpoint saved: 42,176 of 63,978
Checkpoint saved: 42,976 of 63,978
Checkpoint saved: 43,776 of 63,978
Checkpoint saved: 44,576 of 63,978
Checkpoint saved: 45,376 of 63,978
Checkpoint saved: 46,176 of 63,978
Checkpoint saved: 46,976 of 63,978
Checkpoint saved: 47,776 of 63,978
Checkpoint saved: 48,576 of 63,978
Checkpoint saved: 49,376 of 63,978
Checkpoint saved: 50,176 of 63,978
Checkpoint saved: 50,976 of 63,978
Checkpoint saved: 51,776 of 63,97

In [6]:
# Verify the completed predictions
print("Prediction shape:", partial_probabilities.shape)
print("Expected rows:", len(external_test))
print("Contains NaN:", np.isnan(partial_probabilities).any())
print("Contains infinity:", np.isinf(partial_probabilities).any())

# Save the completed external predictions
external_probabilities = partial_probabilities.copy()

np.save(
    "../data/external_test_probabilities.npy",
    external_probabilities
)

print()
print("Final prediction file saved.")
print("Final shape:", external_probabilities.shape)

Prediction shape: (63978, 6)
Expected rows: 63978
Contains NaN: False
Contains infinity: False

Final prediction file saved.
Final shape: (63978, 6)


In [7]:
frozen_thresholds = np.array([
    0.50,  # toxic
    0.25,  # severe_toxic
    0.50,  # obscene
    0.20,  # threat
    0.45,  # insult
    0.20   # identity_hate
])

external_predictions = (
    external_probabilities >= frozen_thresholds
).astype(int)

print("Prediction matrix shape:", external_predictions.shape)
print()
print("Predicted positives by category:")

for index, label in enumerate(label_columns):
    print(
        f"{label}: "
        f"{external_predictions[:, index].sum():,}"
    )

Prediction matrix shape: (63978, 6)

Predicted positives by category:
toxic: 10,516
severe_toxic: 998
obscene: 5,278
threat: 408
insult: 4,255
identity_hate: 1,046


In [8]:
from sklearn.metrics import precision_recall_fscore_support

y_external = external_test[
    label_columns
].to_numpy()

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_external,
        external_predictions,
        average=None,
        zero_division=0
    )
)

category_results = pd.DataFrame({
    "Category": label_columns,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
})

print(category_results.round(4))

print()
print(
    "Macro Precision:",
    round(precision.mean(), 4)
)
print(
    "Macro Recall:",
    round(recall.mean(), 4)
)
print(
    "Macro F1:",
    round(f1.mean(), 4)
)

        Category  Precision  Recall      F1  Support
0          toxic     0.5313  0.9174  0.6729     6090
1   severe_toxic     0.2715  0.7384  0.3971      367
2        obscene     0.5718  0.8177  0.6730     3691
3         threat     0.3750  0.7251  0.4943      211
4         insult     0.6362  0.7899  0.7048     3427
5  identity_hate     0.5239  0.7697  0.6234      712

Macro Precision: 0.485
Macro Recall: 0.793
Macro F1: 0.5942


In [9]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Actual harmful:
# at least one true category label
actual_harmful = (
    y_external.sum(axis=1) > 0
).astype(int)

# Predicted harmful:
# at least one category crossed its frozen threshold
predicted_harmful = (
    external_predictions.sum(axis=1) > 0
).astype(int)

moderation_precision = precision_score(
    actual_harmful,
    predicted_harmful
)

moderation_recall = recall_score(
    actual_harmful,
    predicted_harmful
)

moderation_f1 = f1_score(
    actual_harmful,
    predicted_harmful
)

moderation_cm = confusion_matrix(
    actual_harmful,
    predicted_harmful
)

print("External Moderation-Level Results")
print("---------------------------------")
print(
    "Precision:",
    round(moderation_precision, 4)
)
print(
    "Recall:",
    round(moderation_recall, 4)
)
print(
    "F1:",
    round(moderation_f1, 4)
)

print()
print("Confusion Matrix:")
print(moderation_cm)

tn, fp, fn, tp = moderation_cm.ravel()

print()
print("True Negatives:", f"{tn:,}")
print("False Positives:", f"{fp:,}")
print("False Negatives:", f"{fn:,}")
print("True Positives:", f"{tp:,}")

External Moderation-Level Results
---------------------------------
Precision: 0.5399
Recall: 0.9132
F1: 0.6786

Confusion Matrix:
[[52876  4859]
 [  542  5701]]

True Negatives: 52,876
False Positives: 4,859
False Negatives: 542
True Positives: 5,701


In [10]:
HIDE_CUTOFF = 0.90

max_risk = external_probabilities.max(axis=1)

external_actions = np.where(
    predicted_harmful == 0,
    "ALLOW",
    np.where(
        max_risk >= HIDE_CUTOFF,
        "HIDE_CANDIDATE",
        "REVIEW"
    )
)

policy_results = pd.DataFrame({
    "action": external_actions,
    "actual_harmful": actual_harmful
})

action_summary = (
    policy_results
    .groupby("action")
    .agg(
        total_comments=("actual_harmful", "size"),
        harmful_comments=("actual_harmful", "sum")
    )
)

action_summary["nonharmful_comments"] = (
    action_summary["total_comments"]
    - action_summary["harmful_comments"]
)

action_summary["harmful_rate"] = (
    action_summary["harmful_comments"]
    / action_summary["total_comments"]
)

action_summary["percent_of_all_comments"] = (
    action_summary["total_comments"]
    / len(policy_results)
)

print(action_summary.round(4))

                total_comments  harmful_comments  nonharmful_comments  \
action                                                                  
ALLOW                    53418               542                52876   
HIDE_CANDIDATE            5794              4318                 1476   
REVIEW                    4766              1383                 3383   

                harmful_rate  percent_of_all_comments  
action                                                 
ALLOW                 0.0101                   0.8349  
HIDE_CANDIDATE        0.7453                   0.0906  
REVIEW                0.2902                   0.0745  


### External Moderation Policy Results

The frozen moderation policy automatically classified 83.49% of the external comments as ALLOW, while 16.51% were assigned to REVIEW or HIDE_CANDIDATE.

The ALLOW group remained relatively reliable: only 542 of 53,418 allowed comments (1.01%) were actually harmful. This suggests that the model can still identify a large group of lower-risk comments that may not require routine human review.

However, the external data revealed substantially more false-positive behavior among flagged comments than the internal test set. Of the 5,794 HIDE_CANDIDATE comments, 4,318 (74.53%) were actually harmful, compared with 98.03% on the internal test set. The REVIEW group contained 1,383 harmful comments out of 4,766 (29.02%).

These results show that high model confidence does not guarantee a correct moderation decision on external data. The decrease in precision supports the project's human-in-the-loop design, where HIDE_CANDIDATE represents a moderation recommendation rather than automatic deletion. The external results also demonstrate that moderation workload and false-positive risk may increase when the model encounters data that differs from its development data.